In [0]:
%pip install seaborn sqlalchemy

import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns 

from sqlalchemy import create_engine, text

# EXTRATC

In [0]:
import pandas as pd

# Identifico la ruta del archivo de restaurantes
url_restaurante = "/Workspace/Users/bances27caro@gmail.com/CURSO_PHYTON/Bances-proyecto-etl-delivery./restaurants.csv/restaurants.csv"

# Cargo el archivo
df_restaurantes = pd.read_csv(url_restaurante)

# Visualizó las primeras filas
display(df_restaurantes.head())

In [0]:
import pandas as pd

# Ruta de la carpeta donde están las partes de los menús
ruta_menus = "/Workspace/Users/bances27caro@gmail.com/CURSO_PHYTON/Bances-proyecto-etl-delivery./restaurant-menus.csv"

# Lista con los nombres de los 10 archivos
archivos_menus = [
    f"restaurant_menus_parte_{i}.csv"
    for i in range(1, 11)
]

# Leemos cada archivo y lo guardamos en una lista
lista_menus = []

for archivo in archivos_menus:
    ruta = f"{ruta_menus}/{archivo}"
    df_parte = pd.read_csv(ruta)
    lista_menus.append(df_parte)

# Unimos todas las partes
df_menus = pd.concat(lista_menus, ignore_index=True)

# Mostramos las primeras filas
display(df_menus.head())

In [0]:
# Dimensiones del DataFrame de restaurantes
print(f"df_restaurantes: contiene {df_restaurantes.shape[0]} filas y {df_restaurantes.shape[1]} columnas")

# Dimensiones del DataFrame de menús
print(f"df_menus: contiene {df_menus.shape[0]} filas y {df_menus.shape[1]} columnas")

In [0]:
# Perfil de datos de restaurantes
perfil_restaurantes = pd.DataFrame({
    "Categoria": df_restaurantes.columns,
    "Tipo de dato": df_restaurantes.dtypes.astype(str).values,
    "Valores nulos": df_restaurantes.isna().sum().values
})

display(perfil_restaurantes)

In [0]:
# Perfil de datos de menús
perfil_menus = pd.DataFrame({
    "Categoria": df_menus.columns,
    "Tipo de dato": df_menus.dtypes.astype(str).values,
    "Valores nulos": df_menus.isna().sum().values
})

display(perfil_menus)

# TRANSFORM

In [0]:
mapeo_precios = {
    "$": "Económico",
    "$$": "Moderadamente caro",
    "$$$": "Caro",
    "$$$$": "Muy caro"
}

# Se determina la cantidad de símbolos
df_restaurantes["rango_de_precios"] = df_restaurantes["price_range"].map(mapeo_precios)
df_restaurantes["rango_de_precios"].value_counts()

In [0]:
display(df_restaurantes[["full_address"]].head())

In [0]:
# Se utiliza .str.split() para dividir la columna "full_address" en tres columnas
df_restaurantes[["calle", "ciudad", "estado_zip"]] = (
    df_restaurantes["full_address"].str.split(",", n=2, expand=True)
)
# Se elimina los espacios que puedan quedar antes o después del nombre de la ciudad.
df_restaurantes["ciudad"] = df_restaurantes["ciudad"].str.strip()

display(df_restaurantes[["calle", "ciudad", "estado_zip"]].head())

In [0]:
# Eliminamos los score nulos
df_restaurantes = df_restaurantes[df_restaurantes["score"].notna()]

# Eliminamos los score iguales a 0
df_restaurantes = df_restaurantes[df_restaurantes["score"] != 0]

# Estandarizamos category
df_restaurantes["category"] = (df_restaurantes["category"].str.lower())

display(df_restaurantes[["score", "category"]].head(10))


In [0]:
print("Valores nulos en score:", df_restaurantes["score"].isna().sum())
print("Scores iguales a 0:", (df_restaurantes["score"] == 0).sum())

In [0]:
# Eliminamos los ítems del menú que no tienen precio registrado
df_menus = df_menus[df_menus["price"].notna()]

# Eliminamos el texto " USD"
df_menus["price"] = df_menus["price"].astype(str).str.replace(" USD", "", regex=False)

# Reemplazamos la coma decimal por un punto
df_menus["price"] = df_menus["price"].str.replace(",", ".", regex=False)

# Convertimos la columna a formato decimal
df_menus["price"] = df_menus["price"].astype(float)

# Visualizamos el resultado de la transformación
display(df_menus[["price"]].head(10))

In [0]:
print("Precios nulos:", df_menus["price"].isna().sum())
print("Precios iguales a 0:", (df_menus["price"] == 0).sum())

In [0]:
# Eliminamos los precios nulos
df_menus = df_menus[df_menus["price"].notna()]

# Eliminamos los precios iguales a 0
df_menus = df_menus[df_menus["price"] != 0]

# Visualizamos los datos después de la limpieza
display(df_menus[["price"]].head(10))

In [0]:
print("Precios nulos:", df_menus["price"].isna().sum())
print("Precios iguales a 0:", (df_menus["price"] == 0).sum())

In [0]:
# Visualizamos la columna de calificaciones
display(df_restaurantes[["ratings"]].head(10))

# Filtramos los restaurantes que tienen más de 100 calificaciones
df_restaurantes = df_restaurantes[df_restaurantes["ratings"] > 100]

# Visualizamos el resultado del filtro
display(df_restaurantes[["ratings"]].head(10))

In [0]:
print("Cantidad de restaurantes después del filtro:", len(df_restaurantes))
print("Cantidad mínima de ratings:",df_restaurantes["ratings"].min())

In [0]:
# Realizamos el INNER JOIN mediante el ID del restaurante
df_master = pd.merge(
    df_restaurantes,
    df_menus,
    left_on="id",
    right_on="restaurant_id",
    how="inner"
)

# Visualizamos el DataFrame resultante
display(df_master.head())

In [0]:
# Creamos una copia del DataFrame resultante del INNER JOIN
df_final = df_master.copy()

# Revisamos las columnas disponibles antes de realizar la limpieza
print("Columnas iniciales:")
print(df_final.columns.tolist())

# Eliminamos las columnas que ya no son necesarias
df_final = df_final.drop(
    columns=["restaurant_id", "full_address"]
)

# Cambiamos los nombres para identificar mejor la información
df_final = df_final.rename(
    columns={
        "name_x": "nombre_restaurante",
        "category_x": "categoria_restaurante",
        "name_y": "nombre_plato",
        "category_y": "categoria_menu"
    }
)

# Visualizamos el DataFrame después de la limpieza
display(df_final.head())

# LOAD

In [0]:
# Importamos create_engine desde SQLAlchemy
from sqlalchemy import create_engine

# Creamos el motor de conexión hacia la base de datos SQLite
motor_sql = create_engine("sqlite:///delivery_insights.db")

# Cargamos el DataFrame final en la base de datos
df_final.to_sql(
    name="master_food_data",
    con=motor_sql,
    if_exists="replace",
    index=False
)

print("La tabla master_food_data fue cargada correctamente.")

In [0]:
from sqlalchemy import text

# Consultamos las tablas disponibles en SQLite
consulta_tablas = """
SELECT name
FROM sqlite_master
WHERE type = 'table'
"""

# Ejecutamos la consulta
with motor_sql.connect() as conexion:
    tablas_creadas = pd.read_sql(
        text(consulta_tablas),
        conexion
    )

display(tablas_creadas)

In [0]:
# Consultamos los primeros 5 registros de la tabla
consulta_muestra = """
SELECT *
FROM master_food_data
LIMIT 5
"""

# Ejecutamos la consulta y mostramos los resultados
with motor_sql.connect() as conexion:
    muestra_master = pd.read_sql(
        text(consulta_muestra),
        conexion
    )

display(muestra_master)

# BUSINESS INTELLIGENCE:

In [0]:
from sqlalchemy import create_engine, text
import pandas as pd
import matplotlib.pyplot as plt

# Creamos el motor de conexión hacia la base de datos SQLite
motor_sql = create_engine("sqlite:///delivery_insights.db")

# Establecemos la conexión con la base de datos SQLite
with motor_sql.connect() as conn:
    print("Conexión establecida correctamente con delivery_insights.db")

In [0]:
query_ciudades = """
SELECT 
    ciudad,
    COUNT(DISTINCT id) AS cantidad_restaurantes
FROM master_food_data
GROUP BY ciudad
ORDER BY cantidad_restaurantes DESC
LIMIT 5
"""

# Ejecuto la consulta SQL y almacenamos el resultado
with motor_sql.connect() as conn:
    df_ciudades = pd.read_sql(text(query_ciudades),conn)

display(df_ciudades)

In [0]:
# Ordeno el DataFrame de menor a mayor para que el gráfico
# muestre la ciudad con mayor cantidad en la parte superior
df_grafico = df_ciudades.sort_values(
    "cantidad_restaurantes",
    ascending=True
)

plt.figure(figsize=(9, 5))

plt.barh(
    df_grafico["ciudad"],
    df_grafico["cantidad_restaurantes"],
    color="#905010"

)

plt.xlabel("Cantidad de restaurantes", color="#6A3B2B")
plt.ylabel("Ciudad", color="#6A3B2B")
plt.xticks(color="blue")
plt.yticks(color="blue")
plt.title("Top 5 ciudades con mayor cantidad de restaurantes consolidados", color="#6A3B2B")

plt.grid(axis="both")
plt.gca().set_axisbelow(True)
plt.tight_layout()
plt.show()

Milwaukee y Seattle se concentran la mayor cantidad de restaurantes consolidados, con 122 y 119 respectivamente, superando ampliamente a Lynnwood, Everett y Bellevue. Esto evidencia que la oferta está principalmente concentrada en las dos primeras ciudades.

In [0]:
query_precios = """
SELECT
    rango_de_precios,
    COUNT(*) AS cantidad_restaurantes
FROM master_food_data
GROUP BY rango_de_precios
ORDER BY cantidad_restaurantes DESC
"""

with motor_sql.connect() as conexion:
    df_precios = pd.read_sql(
        text(query_precios),
        conexion
    )

display(df_precios)

In [0]:
plt.figure(figsize=(8, 8))
colors = ['#600000', '#8d1d00', '#bb3a00', '#ff6600']

wedges, texts, autotexts = plt.pie(
    df_precios["cantidad_restaurantes"],
    labels=df_precios["rango_de_precios"],
    colors=colors,
    autopct="%1.1f%%",
    startangle=90
)

for autotext in autotexts:
    autotext.set_color("white")
plt.title("Distribución de restaurantes según rango de precios", color="#6A3B2B")
plt.tight_layout()
plt.show()

El gráfico muestra que la categoría Económico concentra la mayor cantidad de registros, con el 67.5%, seguida de Moderadamente caro con el 23.7%. En cambio, el 8.3% de los registros no presenta una categoría de precio y solo el 0.5% corresponde a la categoría Caro. Esto evidencia un claro predominio de restaurantes con precios económicos frente a las demás categorías.

In [0]:
query_ciudades_precio = """
SELECT
    ciudad,
    AVG(price) AS precio_promedio
FROM master_food_data
WHERE ciudad IN ('Milwaukee', 'Seattle', 'Lynnwood')
GROUP BY ciudad
ORDER BY precio_promedio DESC
"""

with motor_sql.connect() as conexion:
    df_ciudades_precio = pd.read_sql(text(query_ciudades_precio),conexion)
display(df_ciudades_precio)

In [0]:
plt.figure(figsize=(8, 5))

plt.bar(
    df_ciudades_precio["ciudad"],
    df_ciudades_precio["precio_promedio"],
    color="#8d1d00"
)

plt.xlabel("Ciudad", color="blue")
plt.ylabel("Precio promedio ($)", color="blue")
plt.title(
    "Precio promedio de los platos por ciudad",
    color="#6A3B2B"
)
plt.grid(axis="both")
plt.gca().set_axisbelow(True)
plt.tight_layout()
plt.show()

El gráfico muestra el precio promedio de los platos en las tres ciudades más populares. Seattle presenta el mayor precio promedio, con USD 11.98, seguida de Lynnwood con USD 10.82 y Milwaukee con USD 9.53. Si bien los precios presentan diferencias entre las ciudades, se mantienen dentro de un rango relativamente cercano. Estos resultados permiten identificar a Seattle como el mercado con mayor nivel de precios promedio y sirven como referencia para establecer estrategias de menú y precios según cada ciudad.

In [0]:
query_calidad = """
SELECT
    rango_de_precios,
    AVG(score) AS score_promedio
FROM master_food_data
WHERE rango_de_precios IS NOT NULL
GROUP BY rango_de_precios
ORDER BY
    CASE rango_de_precios
        WHEN 'Económico' THEN 1
        WHEN 'Moderadamente caro' THEN 2
        WHEN 'Caro' THEN 3
        WHEN 'Muy caro' THEN 4
    END
"""

with motor_sql.connect() as conexion:
    df_calidad = pd.read_sql(text(query_calidad),conexion)

display(df_calidad)

In [0]:
plt.figure(figsize=(8, 5))

plt.plot(
    df_calidad["rango_de_precios"],
    df_calidad["score_promedio"],
    marker="o",
    linewidth=2,
    color="#8d1d00"
)

plt.xlabel("Rango de precios", color="blue")
plt.ylabel("Score promedio", color="blue")
plt.title("Relación entre rango de precios y calidad promedio", color="#6A3B2B")

plt.ylim(4.0, 5.0)

plt.grid(axis="y", linestyle="--", alpha=0.4)
plt.gca().set_axisbelow(True)

plt.tight_layout()
plt.show()

El gráfico muestra que Moderadamente caro presenta el mayor score promedio, con 4.62, seguido de Económico con 4.61. En cambio, Caro registra el menor score promedio, con 4.43. Esto evidencia que un mayor rango de precios no garantiza una mejor calidad percibida.